In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

try:
    import xlsxwriter                       
    EXCEL_ENGINE = "xlsxwriter"
except ImportError:
    EXCEL_ENGINE = "openpyxl"              

In [2]:
BASE_DIR   = Path(r"C:\Users\LAP TECH\Desktop\DEPI final Project")
FILE_PATH  = BASE_DIR / "DataCoSupplyChainDataset.csv"
OUTPUT_DIR = BASE_DIR / "files"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)   

df = pd.read_csv(FILE_PATH, encoding="cp1252")
rows_before, cols_before = df.shape
print("Shape before cleaning:", df.shape)
df.head(3)

Shape before cleaning: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [3]:
def to_snake(name):
    name = name.strip().lower()
    name = re.sub(r"[^0-9a-z]+", "_", name)
    return name.strip("_")

df.columns = [to_snake(c) for c in df.columns]

df = df.rename(columns={
    "type": "payment_type",
    "order_date_dateorders": "order_date",
    "shipping_date_dateorders": "shipping_date",
})

assert df.columns.is_unique, "في أسماء أعمدة مكررة!"
print(df.columns.tolist())

['payment_type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date', 'shipping_mode']

In [4]:
duplicate_pairs = [
    ("order_profit_per_order", "benefit_per_order"),
    ("order_item_total",       "sales_per_customer"),
    ("product_card_id",        "order_item_cardprod_id"),
    ("category_id",            "product_category_id"),
    ("product_price",          "order_item_product_price"),
    ("customer_id",            "order_customer_id"),
]

for keep, drop in duplicate_pairs:
    assert (df[keep] == df[drop]).all(), f"{keep} و {drop} مش متطابقين!"

df = df.drop(columns=[drop for _, drop in duplicate_pairs])

useless_cols = [
    "customer_email", "customer_password", "product_description",
    "product_status", "order_zipcode", "product_image",
]
df = df.drop(columns=useless_cols)

print("Shape after dropping columns:", df.shape)

Shape after dropping columns: (180519, 41)


In [5]:
text_cols = df.select_dtypes(include=["object", "string"]).columns

for c in text_cols:
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

for c in ["customer_fname", "customer_lname"]:
    df[c] = df[c].str.title()

df["customer_country"] = df["customer_country"].replace({"EE. UU.": "United States"})

In [6]:
# ---- COUNTRY_MAP : اسم الدولة بالإسباني -> الإنجليزي ----
COUNTRY_MAP = {
 'Afganistán':'Afghanistan','Albania':'Albania','Alemania':'Germany','Angola':'Angola','Arabia Saudí':'Saudi Arabia','Argelia':'Algeria',
 'Argentina':'Argentina','Armenia':'Armenia','Australia':'Australia','Austria':'Austria','Azerbaiyán':'Azerbaijan','Bangladés':'Bangladesh',
 'Barbados':'Barbados','Baréin':'Bahrain','Belice':'Belize','Benín':'Benin','Bielorrusia':'Belarus','Bolivia':'Bolivia',
 'Bosnia y Herzegovina':'Bosnia and Herzegovina','Botsuana':'Botswana','Brasil':'Brazil','Bulgaria':'Bulgaria','Burkina Faso':'Burkina Faso',
 'Burundi':'Burundi','Bután':'Bhutan','Bélgica':'Belgium','Camboya':'Cambodia','Camerún':'Cameroon','Canada':'Canada','Chad':'Chad',
 'Chile':'Chile','China':'China','Chipre':'Cyprus','Colombia':'Colombia','Corea del Sur':'South Korea','Costa Rica':'Costa Rica',
 'Costa de Marfil':'Ivory Coast','Croacia':'Croatia','Cuba':'Cuba','Dinamarca':'Denmark','Ecuador':'Ecuador','Egipto':'Egypt',
 'El Salvador':'El Salvador','Emiratos Árabes Unidos':'United Arab Emirates','Eritrea':'Eritrea','Eslovaquia':'Slovakia','Eslovenia':'Slovenia',
 'España':'Spain','Estonia':'Estonia','Etiopía':'Ethiopia','Filipinas':'Philippines','Finlandia':'Finland','Francia':'France','Gabón':'Gabon',
 'Georgia':'Georgia','Ghana':'Ghana','Grecia':'Greece','Guadalupe':'Guadeloupe','Guatemala':'Guatemala','Guayana Francesa':'French Guiana',
 'Guinea':'Guinea','Guinea Ecuatorial':'Equatorial Guinea','Guinea-Bissau':'Guinea-Bissau','Guyana':'Guyana','Haití':'Haiti','Honduras':'Honduras',
 'Hong Kong':'Hong Kong','Hungría':'Hungary','India':'India','Indonesia':'Indonesia','Irak':'Iraq','Irlanda':'Ireland','Irán':'Iran','Israel':'Israel',
 'Italia':'Italy','Jamaica':'Jamaica','Japón':'Japan','Jordania':'Jordan','Kazajistán':'Kazakhstan','Kenia':'Kenya','Kirguistán':'Kyrgyzstan',
 'Kuwait':'Kuwait','Laos':'Laos','Lesoto':'Lesotho','Liberia':'Liberia','Libia':'Libya','Lituania':'Lithuania','Luxemburgo':'Luxembourg',
 'Líbano':'Lebanon','Macedonia':'North Macedonia','Madagascar':'Madagascar','Malasia':'Malaysia','Mali':'Mali','Marruecos':'Morocco',
 'Martinica':'Martinique','Mauritania':'Mauritania','Moldavia':'Moldova','Mongolia':'Mongolia','Montenegro':'Montenegro','Mozambique':'Mozambique',
 'Myanmar (Birmania)':'Myanmar','México':'Mexico','Namibia':'Namibia','Nepal':'Nepal','Nicaragua':'Nicaragua','Nigeria':'Nigeria','Noruega':'Norway',
 'Nueva Zelanda':'New Zealand','Níger':'Niger','Omán':'Oman','Pakistán':'Pakistan','Panamá':'Panama','Papúa Nueva Guinea':'Papua New Guinea',
 'Paraguay':'Paraguay','Países Bajos':'Netherlands','Perú':'Peru','Polonia':'Poland','Portugal':'Portugal','Qatar':'Qatar','Reino Unido':'United Kingdom',
 'República Centroafricana':'Central African Republic','República Checa':'Czech Republic','República Democrática del Congo':'Democratic Republic of the Congo',
 'República Dominicana':'Dominican Republic','República de Gambia':'Gambia','República del Congo':'Republic of the Congo','Ruanda':'Rwanda',
 'Rumania':'Romania','Rusia':'Russia','Senegal':'Senegal','Serbia':'Serbia','Sierra Leona':'Sierra Leone','Singapur':'Singapore','Siria':'Syria',
 'Somalia':'Somalia','Sri Lanka':'Sri Lanka','Suazilandia':'Eswatini','SudAfrica':'South Africa','Sudán':'Sudan','Sudán del Sur':'South Sudan',
 'Suecia':'Sweden','Suiza':'Switzerland','Surinam':'Suriname','Sáhara Occidental':'Western Sahara','Tailandia':'Thailand','Taiwán':'Taiwan',
 'Tanzania':'Tanzania','Tayikistán':'Tajikistan','Togo':'Togo','Trinidad y Tobago':'Trinidad and Tobago','Turkmenistán':'Turkmenistan',
 'Turquía':'Turkey','Túnez':'Tunisia','Ucrania':'Ukraine','Uganda':'Uganda','Estados Unidos':'United States','Uruguay':'Uruguay',
 'Uzbekistán':'Uzbekistan','Venezuela':'Venezuela','Vietnam':'Vietnam','Yemen':'Yemen','Yibuti':'Djibouti','Zambia':'Zambia','Zimbabue':'Zimbabwe',
}

# ---- STATE_MAP : اسم الـ state بالإسباني -> الإنجليزي (اللي مش موجود هنا بيفضل زي ما هو) ----
STATE_MAP = {
    # United States
    "Misuri": "Missouri", "Pensilvania": "Pennsylvania", "Nueva York": "New York",
    "Carolina del Norte": "North Carolina", "Carolina del Sur": "South Carolina",
    "Nueva Jersey": "New Jersey", "Virginia Occidental": "West Virginia",
    "Misisipi": "Mississippi", "Nuevo México": "New Mexico", "Luisiana": "Louisiana",
    "Nuevo Hampshire": "New Hampshire", "Distrito de Columbia": "District of Columbia",
    "Dakota del Sur": "South Dakota", "Dakota del Norte": "North Dakota",
    # Canada / Mexico / Latin America
    "Terranova": "Newfoundland and Labrador", "Columbia británica": "British Columbia",
    "Nueva Escocia": "Nova Scotia", "Distrito Federal": "Mexico City", "México": "State of Mexico",
    "Marañón": "Maranhao", "Espíritu Santo": "Espirito Santo", "Río Grande del Norte": "Rio Grande do Norte",
    "Mato Grosso del Sur": "Mato Grosso do Sul", "Buenos Aires (provincia)": "Buenos Aires Province",
    "Lima (ciudad)": "Lima", "Santiago de Chile": "Santiago Metropolitan", "Bío-Bío": "Biobio",
    "Magallanes y Antártica Chilena": "Magallanes and Chilean Antarctica",
    "Distrito Capital": "Capital District", "Isla de la Juventud": "Isle of Youth",
    "Artibonito": "Artibonite", "Guadalupe": "Guadeloupe", "Martinica": "Martinique",
    "Französisch-Guayana": "French Guiana", "Distrito de Cayo": "Cayo District",
    "San Miguel": "Saint Michael", "Santa Caterina": "Saint Catherine",
    # Spain / Portugal
    "Cataluña": "Catalonia", "Valenciana": "Valencia", "Islas Baleares": "Balearic Islands",
    "País Vasco": "Basque Country", "Andalucía": "Andalusia", "Castilla y León": "Castile and Leon",
    "Castilla-La Mancha": "Castile-La Mancha", "Navarra": "Navarre", "Lisboa": "Lisbon",
    # Germany / Austria / Switzerland / Benelux
    "Renania del Norte-Westfalia": "North Rhine-Westphalia", "Sajonia-Anhalt": "Saxony-Anhalt",
    "Baja Sajonia": "Lower Saxony", "Hamburgo": "Hamburg", "Baden-Wurtemberg": "Baden-Wurttemberg",
    "Sajonia": "Saxony", "Brandenburgo": "Brandenburg", "Renania-Palatinado": "Rhineland-Palatinate",
    "Mecklemburgo-Pomerania Occidental": "Mecklenburg-Western Pomerania", "Turingia": "Thuringia",
    "Sarre": "Saarland", "Salzburgo": "Salzburg", "Tirol": "Tyrol", "Viena": "Vienna",
    "Carintia": "Carinthia", "Alta Austria": "Upper Austria", "Stiria": "Styria",
    "Berna": "Bern", "Ginebra": "Geneva", "San Galo": "St. Gallen", "Basilea-Ciudad": "Basel-Stadt",
    "Lucerna": "Lucerne", "Lieja": "Liege", "Limburgo": "Limburg", "Henao": "Hainaut",
    "Amberes": "Antwerp", "Bruselas": "Brussels", "Flanders Oriental": "East Flanders",
    "Brabante Flamenco": "Flemish Brabant", "Flanders Occidental": "West Flanders",
    "Holanda Septentrional": "North Holland", "Holanda Meridional": "South Holland",
    "Güeldres": "Gelderland", "Brabante Septentrional": "North Brabant", "Frisia": "Friesland",
    "Zelanda": "Zeeland", "Luxemburgo": "Luxembourg",
    # France / Italy
    "Isla de Francia": "Ile-de-France", "Países del Loira": "Pays de la Loire",
    "Alsacia-Champaña-Ardenas-Lorena": "Alsace-Champagne-Ardenne-Lorraine",
    "Provenza-Alpes-Costa Azul": "Provence-Alpes-Cote d'Azur",
    "Norte-Paso de Calais-Picardía": "Nord-Pas-de-Calais-Picardy",
    "Aquitania-Lemosín-Poitou-Charentes": "Aquitaine-Limousin-Poitou-Charentes",
    "Languedoc-Rosellón-Mediodía-Pirineos": "Languedoc-Roussillon-Midi-Pyrenees",
    "Bretaña": "Brittany", "Centro-Valle de Loira": "Centre-Val de Loire", "Normandía": "Normandy",
    "Auvernia-Ródano-Alpes": "Auvergne-Rhone-Alpes", "Auvernia": "Auvergne",
    "Borgoña-Franco Condado": "Burgundy-Franche-Comte", "Corse": "Corsica",
    "Friuli-Venecia Julia": "Friuli-Venezia Giulia", "Trentino-Alto Adigio": "Trentino-Alto Adige",
    "Sicilia": "Sicily", "Toscana": "Tuscany", "Lombardía": "Lombardy", "Piamonte": "Piedmont",
    "Cerdeña": "Sardinia", "Lacio": "Lazio", "Emilia-Romaña": "Emilia-Romagna", "Abruzos": "Abruzzo",
    # UK / Ireland / Nordics / Eastern Europe
    "Inglaterra": "England", "Escocia": "Scotland", "Gales": "Wales", "Dublín del Sur": "South Dublin",
    "Región Capital": "Capital Region", "Dinamarca Meridional": "Southern Denmark",
    "Jutlandia Central": "Central Jutland", "Selandia": "Zealand",
    "Finlandia del Sudoeste": "Southwest Finland", "Estocolmo": "Stockholm", "Upsala": "Uppsala",
    "Escania": "Skane", "Gran Polonia": "Greater Poland", "Podlaquia": "Podlaskie",
    "Polonia Menor": "Lesser Poland", "Moravia-Silesia": "Moravian-Silesian", "Praga": "Prague",
    "Sofía-Ciudad": "Sofia City", "Constanza": "Constanta", "Split-Dalmacia": "Split-Dalmatia",
    "Serbia Central": "Central Serbia", "Federación de Bosnia y Herzegovina": "Federation of Bosnia and Herzegovina",
    "Liubliana": "Ljubljana", "Vilna": "Vilnius", "Singapur": "Singapore", "Túnez": "Tunis",
    "Yibuti": "Djibouti", "Gómel": "Gomel", "Moscú": "Moscow", "Bashkortostán": "Bashkortostan",
    "Daguestán": "Dagestan", "Cheliábinsk": "Chelyabinsk", "Astracán": "Astrakhan",
    "Kaliningrado": "Kaliningrad", "Stávropol": "Stavropol", "Arjánguelsk": "Arkhangelsk",
    "Vladímir": "Vladimir", "Bélgorod": "Belgorod", "Briansk": "Bryansk", "Vorónezh": "Voronezh",
    "Nóvgorod": "Novgorod", "Yevrey": "Jewish Autonomous Oblast", "Kabardia-Balkaria": "Kabardino-Balkaria",
    "Járkov": "Kharkiv", "Vínnytsia": "Vinnytsia", "Zaporiyia": "Zaporizhzhia", "Volinia": "Volyn",
    "Ternópil": "Ternopil", "Zhytómyr": "Zhytomyr", "Jersón": "Kherson",
    "Imericia": "Imereti", "Tiflis": "Tbilisi",
    # Australia / New Zealand
    "Nueva Gales del Sur": "New South Wales", "Australia Occidental": "Western Australia",
    "Territorio de la Capital Australiana": "Australian Capital Territory",
    "Australia del Sur": "South Australia", "Territorio del Norte": "Northern Territory",
    # Asia
    "Shanghái": "Shanghai", "Pekín": "Beijing", "Mongolia Interior": "Inner Mongolia",
    "Seúl": "Seoul", "Chungcheong del Sur": "South Chungcheong", "Tokio": "Tokyo",
    "Rajastán": "Rajasthan", "Guyarat": "Gujarat", "Bengala Occidental": "West Bengal",
    "Punyab": "Punjab", "Jammu y Cachemira": "Jammu and Kashmir",
    "Java Occidental": "West Java", "Célebes Septentrional": "North Sulawesi",
    "Sumatra Septentrional": "North Sumatra", "Java Central": "Central Java",
    "Kalimantan Oriental": "East Kalimantan", "Java Oriental": "East Java", "Yakarta": "Jakarta",
    "Célebes Suroriental": "Southeast Sulawesi", "Sumatra Meridional": "South Sumatra",
    "Borneo Occidental": "West Kalimantan", "Sumatra Occidental": "West Sumatra",
    "Borneo Meridional": "South Kalimantan", "Célebes Central": "Central Sulawesi",
    "Nusa Tenggara Oriental": "East Nusa Tenggara", "Nusatenggara Occidental": "West Nusa Tenggara",
    "Capital Nacional": "National Capital Region", "Mindanao Septentrional": "Northern Mindanao",
    "Visayas Occidental": "Western Visayas", "Luzón Central": "Central Luzon",
    "Trengganu": "Terengganu", "Nom Pen": "Phnom Penh", "Rangún": "Yangon",
    "Jaiber Pajtunjuá": "Khyber Pakhtunkhwa", "Sind": "Sindh", "Baluchistán": "Balochistan",
    "Territorio de la Capital Federal": "Islamabad Capital Territory",
    "Ulán Bator": "Ulaanbaatar", "Prefectura de Vientián": "Vientiane Prefecture",
    "Ciudad Ho Chi Minh": "Ho Chi Minh City", "Th? Dô Hà N?i": "Hanoi", "T?nh C?n Th?": "Can Tho",
    "Trebisonda": "Trabzon", "Estambul": "Istanbul", "Kahramanmara?": "Kahramanmaras",
    "Daca": "Dhaka", "Baki": "Baku", "Karagandá": "Karaganda", "Ciudad de Almatý": "Almaty",
    "Kostanái": "Kostanay", "Astaná": "Astana", "Asjabad": "Ashgabat",
    "Surjandarín": "Surxondaryo", "Kashkadar": "Qashqadaryo", "Andiján": "Andijan",
    "Gy?r": "Gyor", "Klaip?da": "Klaipeda", "Bra?ov": "Brasov",
    # Middle East
    "Riad": "Riyadh", "Meca": "Mecca", "Jizán": "Jazan", "Bagdad": "Baghdad", "Basora": "Basra",
    "Cadisia": "Qadisiyah", "Nínive": "Nineveh", "Babilonia": "Babylon", "Di Car": "Dhi Qar",
    "Saladino": "Saladin", "Mesena": "Maysan", "Diala": "Diyala", "Guilán": "Gilan",
    "Juzestán": "Khuzestan", "Sistán y Baluchistán": "Sistan and Baluchestan", "Zanyán": "Zanjan",
    "Jorasán Razaví": "Razavi Khorasan", "Isfahán": "Isfahan", "Teherán": "Tehran",
    "Ardebil": "Ardabil", "Kurdistán": "Kurdistan", "Jorasán del Sur": "South Khorasan",
    "Jerusalén": "Jerusalem", "Amán": "Amman", "Líbano del Norte": "North Lebanon",
    "Mascate": "Muscat", "Ras al-Jaima": "Ras Al Khaimah", "Ar Raqa": "Ar Raqqah",
    "Al Hudayda": "Al Hudaydah", "Bengasi": "Benghazi", "Misurata": "Misrata",
    "El Cairo": "Cairo", "Alejandría": "Alexandria", "Fayún": "Faiyum", "Menia": "Minya",
    "Puerto Saíd": "Port Said", "Kafr el Sheij": "Kafr El Sheikh", "Asiut": "Asyut",
    "Constantina": "Constantine", "Bugía": "Bejaia",
    # Africa
    "Gran Casablanca": "Greater Casablanca", "Garb-Chrarda-Beni Hsen": "Gharb-Chrarda-Beni Hssen",
    "Fez-Bulmán": "Fes-Boulemane", "Rabat-Salé-Zemur-Zaer": "Rabat-Sale-Zemmour-Zaer",
    "Marrakech-Tensift-Al Hauz": "Marrakesh-Tensift-El Haouz", "Chauía-Uardiga": "Chaouia-Ouardigha",
    "Mequinez-Tafilalet": "Meknes-Tafilalet", "Taza-Alhucemas-Taunat": "Taza-Al Hoceima-Taounate",
    "Tánger-Tetuán": "Tangier-Tetouan", "Dukala-Abda": "Doukkala-Abda", "Sus-Masa-Draa": "Souss-Massa-Draa",
    "Guelmim-Esmara": "Guelmim-Es Semara", "El Aaiún": "Laayoune", "Cabo Oriental": "Eastern Cape",
    "Cabo Occidental": "Western Cape", "Noroeste": "North West", "Estado Libre": "Free State",
    "Jartum": "Khartoum", "Mar Rojo": "Red Sea", "Nilo Blanco": "White Nile", "Kasala": "Kassala",
    "Ecuatoria Central": "Central Equatoria", "Gran Accra": "Greater Accra", "Costa": "Coast",
    "Gran Valle del Rift": "Rift Valley", "Marítima": "Maritime", "Ríos": "Rivers",
    "Provincia Oriental": "Orientale", "Kasai Occidental": "West Kasai", "Kasai Oriental": "East Kasai", "Kivu del Sur": "South Kivu",
    "Bajo Congo": "Bas-Congo", "Bisáu": "Bissau", "Buyumbura Mairie": "Bujumbura Mairie",
    "Cidade De Maputo": "Maputo City", "Manicalandia": "Manicaland",
    "Mashonalandia Occidental": "Mashonaland West", "Gobolka Woqooyi Galbeed": "Woqooyi Galbeed",
    "Demerara-Berbice Superior": "Upper Demerara-Berbice", "Departamento Atlántico": "Atlantique",
    "Sudeste": "South-East", "Centro": "Centre", "Ouest": "West", "Nord": "North", "Sud": "South",
    "Est": "East", "Ouest del Sur": "South-West", "Nord-Ouest": "North-West", "Extreme-Nord": "Far North",
    "Septentrional": "Northern", "Meridional": "Southern", "Occidental": "Western", "Oriental": "Eastern",
}

# حالات خاصة: نفس الاسم الإسباني معناه حاجة تانية حسب الدولة
STATE_BY_COUNTRY = {
    ("Brasil", "Distrito Federal"): "Federal District",
    ("República Dominicana", "Santiago de Chile"): "Santiago",
}

# ---- CITY_MAP : اسم المدينة بالإسباني -> الإنجليزي ----
CITY_MAP = {
    "Berlín": "Berlin", "Brandenburgo": "Brandenburg", "Hamburgo": "Hamburg", "Pekín": "Beijing",
    "Shanghái": "Shanghai", "Seúl": "Seoul", "Tokio": "Tokyo", "Yakarta": "Jakarta",
    "Rangún": "Yangon", "Nom Pen": "Phnom Penh", "Daca": "Dhaka", "Viena": "Vienna",
    "Salzburgo": "Salzburg", "Amberes": "Antwerp", "Bruselas": "Brussels", "Praga": "Prague",
    "Bagdad": "Baghdad", "Jerusalén": "Jerusalem", "Teherán": "Tehran", "Hamadán": "Hamadan",
    "Qazvín": "Qazvin", "Semnán": "Semnan", "Zanyán": "Zanjan", "Ardebil": "Ardabil",
    "Mascate": "Muscat", "Dublín": "Dublin", "Líbano": "Lebanon", "Luxemburgo": "Luxembourg",
    "Liubliana": "Ljubljana", "Vilna": "Vilnius", "Estocolmo": "Stockholm", "Upsala": "Uppsala",
    "Ginebra": "Geneva", "Lucerna": "Lucerne", "San Galo": "St. Gallen", "Túnez": "Tunis",
    "Singapur": "Singapore", "Yibuti": "Djibouti", "Estambul": "Istanbul", "Trebisonda": "Trabzon",
    "Jartum": "Khartoum", "Kasala": "Kassala", "Bugía": "Bejaia", "Constantina": "Constantine",
    "Orán": "Oran", "Jizán": "Jazan", "Fayún": "Faiyum", "Menia": "Minya", "Tiflis": "Tbilisi",
    "Bisáu": "Bissau", "Gómel": "Gomel", "Ar Raqa": "Ar Raqqah", "Asjabad": "Ashgabat",
    "El Aaiún": "Laayoune", "Ulan Bator": "Ulaanbaatar", "Chisináu": "Chisinau",
    "Astaná": "Astana", "Karagandá": "Karaganda", "Kostanái": "Kostanay",
    "Astracán": "Astrakhan", "Briansk": "Bryansk", "Cheliábinsk": "Chelyabinsk",
    "Kaliningrado": "Kaliningrad", "Vladímir": "Vladimir", "Vorónezh": "Voronezh",
    "Jersón": "Kherson", "Járkov": "Kharkiv", "Vínnytsia": "Vinnytsia", "Zaporiyia": "Zaporizhzhia",
    "Zhytómyr": "Zhytomyr", "Ternópil": "Ternopil", "Santiago de Chile": "Santiago",
    "Constanza": "Constanta",
    # أسماء فيها "?" جاية بايظة من الملف الأصلي (رجّعتها لشكلها الصح)
    "Cox’s B?z?r": "Cox's Bazar", "Aew?l-li": "Aewol-li", "Gy?r": "Gyor", "Klaip?da": "Klaipeda",
    "Bra?ov": "Brasov", "Kahramanmara?": "Kahramanmaras",
}


In [7]:
def to_ascii(text):
    """بيشيل علامات الحروف: Bogotá -> Bogota"""
    text = unicodedata.normalize("NFKD", text.replace("’", "'"))
    return text.encode("ascii", "ignore").decode("ascii")


missing = set(df["order_country"].unique()) - set(COUNTRY_MAP)
assert not missing, f"دول مش موجودة في COUNTRY_MAP: {missing}"

df["order_state"] = [
    STATE_BY_COUNTRY.get((country, state), STATE_MAP.get(state, state))
    for country, state in zip(df["order_country"], df["order_state"])
]
df["order_country"] = df["order_country"].replace(COUNTRY_MAP)

df["order_city"] = df["order_city"].replace(CITY_MAP)

for col in ["order_state", "order_city"]:
    mapping = {v: to_ascii(v) for v in df[col].unique()}
    df[col] = df[col].map(mapping)
    df[col] = df[col].str.replace("?", "", regex=False).str.strip()

print("Countries :", df["order_country"].nunique())
print("States    :", df["order_state"].nunique())
print("Cities    :", df["order_city"].nunique())
df[["order_country", "order_state", "order_city"]].drop_duplicates().head(8)

Countries : 164
States    : 1089
Cities    : 3594


,order_country,order_state,order_city
0,Indonesia,West Java,Bekasi
1,India,Rajasthan,Bikaner
3,Australia,Queensland,Townsville
5,Australia,Queensland,Toowoomba
6,China,Guangdong,Guangzhou
10,Japan,Tokyo,Tokyo
11,Indonesia,North Sulawesi,Manado
13,India,Maharashtra,Sangli


In [8]:
df["customer_zipcode"] = df["customer_zipcode"].astype("Int64").astype("string").str.zfill(5)

shifted = df["customer_state"].str.fullmatch(r"\d{5}")
print("Shifted rows found:", int(shifted.sum()))

df.loc[shifted, "customer_zipcode"] = df.loc[shifted, "customer_state"]
df.loc[shifted, "customer_state"]   = df.loc[shifted, "customer_city"]
df.loc[shifted, "customer_city"]    = "Unknown"

df[shifted][["customer_id", "customer_city", "customer_state", "customer_zipcode"]]

Shifted rows found: 3


,customer_id,customer_city,customer_state,customer_zipcode
35704,14577,Unknown,CA,95758
46440,17171,Unknown,CA,95758
82511,14046,Unknown,CA,91732


In [9]:
df["customer_lname"] = df["customer_lname"].fillna("Unknown")

print("Missing values left:", int(df.isna().sum().sum()))

Missing values left: 0


In [10]:
df["order_date"]    = pd.to_datetime(df["order_date"],    format="%m/%d/%Y %H:%M")
df["shipping_date"] = pd.to_datetime(df["shipping_date"], format="%m/%d/%Y %H:%M")

print(df["order_date"].min(), "→", df["order_date"].max())

2015-01-01 00:00:00 → 2018-01-31 23:38:00


In [11]:
money_cols = [
    "order_profit_per_order", "order_item_total", "order_item_discount",
    "product_price", "sales",
]
df[money_cols] = df[money_cols].round(2)
df["order_item_profit_ratio"] = df["order_item_profit_ratio"].round(2)

In [12]:
df["order_year"]        = df["order_date"].dt.year
df["order_month"]       = df["order_date"].dt.month
df["order_month_name"]  = df["order_date"].dt.month_name()
df["order_day_name"]    = df["order_date"].dt.day_name()

df["shipping_delay_days"] = df["days_for_shipping_real"] - df["days_for_shipment_scheduled"]

df[["order_date", "order_year", "order_month_name", "order_day_name", "shipping_delay_days"]].head()

,order_date,order_year,order_month_name,order_day_name,shipping_delay_days
0,2018-01-31 22:56:00,2018,January,Wednesday,-1
1,2018-01-13 12:27:00,2018,January,Saturday,1
2,2018-01-13 12:06:00,2018,January,Saturday,0
3,2018-01-13 11:45:00,2018,January,Saturday,-1
4,2018-01-13 11:24:00,2018,January,Saturday,-2


In [13]:
dups = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", dups)

Duplicate rows removed: 0


In [14]:
text_cols = df.select_dtypes(include=["object", "string"]).columns

assert df.isna().sum().sum() == 0, "لسه فيه قيم فاضية"

assert df.duplicated().sum() == 0, "لسه فيه صفوف مكررة"
assert df["order_item_id"].is_unique, "order_item_id فيه تكرار"

for c in text_cols:
    assert (df[c] == df[c].str.strip()).all(), f"مسافات زيادة في {c}"
    assert not df[c].str.contains(r"\s{2,}", regex=True).any(), f"مسافات مزدوجة في {c}"
    assert not df[c].str.contains(r"[\x00-\x1f\x7f-\x9f]", regex=True).any(), f"رموز مخفية في {c}"

geo_cols = ["order_country", "order_state", "order_city"]
for c in geo_cols:
    assert not df[c].str.contains(r"[^\x00-\x7F]", regex=True).any(), f"حروف مش إنجليزي في {c}"
    assert not df[c].str.contains("?", regex=False).any(), f"لسه فيه علامة ? في {c}"
spanish_left = set(df["order_country"]) & {k for k, v in COUNTRY_MAP.items() if k != v}
assert not spanish_left, f"دول لسه بالإسباني: {spanish_left}"
assert df["order_country"].nunique() == len(set(COUNTRY_MAP.values()))

assert pd.api.types.is_datetime64_any_dtype(df["order_date"])
assert pd.api.types.is_datetime64_any_dtype(df["shipping_date"])

assert (df["order_date"] <= df["shipping_date"]).all(), "تاريخ الشحن قبل تاريخ الطلب"
assert (df["order_item_quantity"] >= 1).all()
assert (df["sales"] > 0).all()
assert df["order_item_discount_rate"].between(0, 1).all()
assert (df["days_for_shipping_real"] >= 0).all()
assert df["latitude"].between(-90, 90).all()
assert df["longitude"].between(-180, 180).all()

assert df["customer_zipcode"].str.fullmatch(r"\d{5}").all(), "zipcode مش 5 أرقام"
assert df["customer_state"].str.fullmatch(r"[A-Z]{2}").all(), "في state مش حرفين"

assert df["late_delivery_risk"].isin([0, 1]).all()
assert df["delivery_status"].nunique() == 4
assert "South of USA" in set(df["order_region"]), "اسم المنطقة لسه فيه مسافتين"

print("✅ كل الفحوصات عدّت — الداتا نضيفة 100%")

✅ كل الفحوصات عدّت — الداتا نضيفة 100%


In [15]:
print(f"Rows   : {rows_before:,} → {len(df):,}")
print(f"Columns: {cols_before} → {df.shape[1]}")
print(f"Nulls  : {int(df.isna().sum().sum())}")
print(f"Dupes  : {int(df.duplicated().sum())}")
print()
df.info()

Rows   : 180,519 → 180,519
Columns: 53 → 46
Nulls  : 0
Dupes  : 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 46 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   payment_type                 180519 non-null  object        
 1   days_for_shipping_real       180519 non-null  int64         
 2   days_for_shipment_scheduled  180519 non-null  int64         
 3   delivery_status              180519 non-null  object        
 4   late_delivery_risk           180519 non-null  int64         
 5   category_id                  180519 non-null  int64         
 6   category_name                180519 non-null  object        
 7   customer_city                180519 non-null  object        
 8   customer_country             180519 non-null  object        
 9   customer_fname               180519 non-null  object        
 10  customer_id              

In [16]:
OUTPUT_PATH = OUTPUT_DIR / "DataCo_cleaned.xlsx"
df.to_excel(OUTPUT_PATH, index=False, sheet_name="DataCo_cleaned", engine=EXCEL_ENGINE)
print("Saved:", OUTPUT_PATH, "→", df.shape)

df.head()

Saved: C:\Users\LAP TECH\Desktop\DEPI final Project\files\DataCo_cleaned.xlsx → (180519, 46)


,payment_type,days_for_shipping_real,days_for_shipment_scheduled,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_date,order_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,product_card_id,product_name,product_price,shipping_date,shipping_mode,order_year,order_month,order_month_name,order_day_name,shipping_delay_days
0,DEBIT,3,4,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Cally,20755,Holloway,Consumer,PR,5365 Noble Nectar Island,00725,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,2018-01-31 22:56:00,77202,13.11,0.04,180517,0.29,1,327.75,314.64,91.25,Southeast Asia,West Java,COMPLETE,1360,Smart watch,327.75,2018-02-03 22:56:00,Standard Class,2018,1,January,Wednesday,-1
1,TRANSFER,5,4,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,Irene,19492,Luna,Consumer,PR,2679 Rustic Loop,00725,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,2018-01-13 12:27:00,75939,16.39,0.05,179254,-0.80,1,327.75,311.36,-249.09,South Asia,Rajasthan,PENDING,1360,Smart watch,327.75,2018-01-18 12:27:00,Standard Class,2018,1,January,Saturday,1
2,CASH,4,4,Shipping on time,0,73,Sporting Goods,San Jose,United States,Gillian,19491,Maldonado,Consumer,CA,8510 Round Bear Gate,95125,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,2018-01-13 12:06:00,75938,18.03,0.06,179253,-0.80,1,327.75,309.72,-247.78,South Asia,Rajasthan,CLOSED,1360,Smart watch,327.75,2018-01-17 12:06:00,Standard Class,2018,1,January,Saturday,0
3,DEBIT,3,4,Advance shipping,0,73,Sporting Goods,Los Angeles,United States,Tana,19490,Tate,Home Office,CA,3200 Amber Bend,90027,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,2018-01-13 11:45:00,75937,22.94,0.07,179252,0.08,1,327.75,304.81,22.86,Oceania,Queensland,COMPLETE,1360,Smart watch,327.75,2018-01-16 11:45:00,Standard Class,2018,1,January,Saturday,-1
4,PAYMENT,2,4,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Orli,19489,Hendricks,Corporate,PR,8671 Iron Anchor Corners,00725,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,2018-01-13 11:24:00,75936,29.50,0.09,179251,0.45,1,327.75,298.25,134.21,Oceania,Queensland,PENDING_PAYMENT,1360,Smart watch,327.75,2018-01-15 11:24:00,Standard Class,2018,1,January,Saturday,-2
